# Day 02: Calculus & Gradients — How Models Learn

**Goal:** Understand derivatives, gradients, and gradient descent — the algorithm that trains every LLM.

Yesterday we learned: `output = input @ weights + bias` — but the weights start as **random numbers**. Today we answer the big question: **how does the model figure out the right weights?**

### The Training Loop (in plain English)

Every model learns the same way:

1. **Forward pass** — feed data in, get a prediction out
2. **Compute loss** — measure how wrong the prediction is (a single number)
3. **Compute gradients** — for each weight, ask: "should I increase or decrease this to reduce the loss?"
4. **Update weights** — nudge each weight a tiny bit in the right direction
5. **Repeat** millions of times until the loss is small

Step 3 is where calculus comes in. But the core idea is simple: **a derivative tells you which direction is downhill.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. What is a Derivative? (Slope = Rate of Change)

Imagine you're standing on a hill. The derivative tells you **how steep the ground is under your feet**.

- **Positive derivative** → the ground slopes UP to the right → go LEFT to go downhill
- **Negative derivative** → the ground slopes DOWN to the right → go RIGHT to go downhill
- **Zero derivative** → the ground is FLAT → you might be at the bottom!

### In math terms:
The derivative of `f(x)` at a point answers: **"if I increase x by a tiny bit, how much does f(x) change?"**

```
f(x) = x²   →   derivative f'(x) = 2x

At x = -3: f'(-3) = -6  → function is going DOWN (slope is negative)
At x =  0: f'(0)  =  0  → function is FLAT (this is the minimum!)
At x =  2: f'(2)  =  4  → function is going UP (slope is positive)
```

### Why do we care?
In ML, `f` is our **loss function** (how wrong the model is) and `x` is a **weight**. The derivative tells us: "should I increase or decrease this weight to make the model less wrong?"

Let's visualize this:

In [ ]:
# Let's visualize f(x) = x² and its derivative f'(x) = 2x

def f(x):
    return x ** 2

def f_derivative(x):
    return 2 * x

x = np.linspace(-4, 4, 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Plot the function
ax1.plot(x, f(x), 'b-', linewidth=2)
ax1.set_title("f(x) = x²")
ax1.set_xlabel("x")
ax1.set_ylabel("f(x)")
ax1.axhline(y=0, color='k', linewidth=0.5)
ax1.axvline(x=0, color='k', linewidth=0.5)
ax1.grid(True, alpha=0.3)

# Mark some points and their slopes
for xi, color in [(-3, 'red'), (0, 'green'), (2, 'orange')]:
    ax1.plot(xi, f(xi), 'o', color=color, markersize=10)
    # Draw tangent line
    slope = f_derivative(xi)
    tangent_x = np.linspace(xi - 1.5, xi + 1.5, 50)
    tangent_y = f(xi) + slope * (tangent_x - xi)
    ax1.plot(tangent_x, tangent_y, '--', color=color, alpha=0.7, 
             label=f"x={xi}, slope={slope}")

ax1.legend()

# Plot the derivative
ax2.plot(x, f_derivative(x), 'r-', linewidth=2)
ax2.set_title("f'(x) = 2x (the derivative)")
ax2.set_xlabel("x")
ax2.set_ylabel("f'(x) = slope")
ax2.axhline(y=0, color='k', linewidth=0.5)
ax2.axvline(x=0, color='k', linewidth=0.5)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("At x=-3: slope = -6 (going DOWN → move right to reduce f)")
print("At x= 0: slope =  0 (FLAT → this is the minimum!)")
print("At x= 2: slope =  4 (going UP → move left to reduce f)")

### What the plot shows:

**Left plot (the function):**
- The U-shaped curve is `f(x) = x²` — think of it as a bowl
- The dashed lines are **tangent lines** — they show the slope at that point
- At x = -3 (red): the tangent slopes down-right → derivative is negative (-6)
- At x = 0 (green): the tangent is flat → derivative is zero (the minimum!)
- At x = 2 (orange): the tangent slopes up-right → derivative is positive (4)

**Right plot (the derivative):**
- This is just `f'(x) = 2x` — a straight line
- Where this line crosses zero → the original function has its minimum
- The sign (+/-) tells you which direction is "downhill"

**Key insight:** If we're at x = 2 and the derivative is positive (+4), we should **decrease** x (go left) to reach the minimum. If we're at x = -3 and the derivative is negative (-6), we should **increase** x (go right). In both cases: **go opposite to the derivative**.

## 2. Numerical Derivative — Computing Slope Without Calculus

Here's the good news: **you don't need to know any calculus formulas to compute a derivative.** Just measure it!

The idea is simple:
1. Evaluate `f(x)` 
2. Nudge x by a tiny amount `h` (like 0.00001) and evaluate `f(x+h)`
3. The slope = how much f changed / how much x changed

```
f'(x) ≈ (f(x + h) - f(x - h)) / (2 × h)
```

It's like measuring the steepness of a hill by taking one step forward and one step back, then checking how much your elevation changed.

**Why both sides (x+h and x-h)?** Using both sides gives a more accurate estimate — errors from one side cancel out the other. This is called the "central difference" method.

In [ ]:
def numerical_derivative(f, x, h=1e-5):
    """Compute derivative of f at x using tiny nudge."""
    return (f(x + h) - f(x - h)) / (2 * h)

# Test on f(x) = x²  →  f'(x) should be 2x
def f(x):
    return x ** 2

# Compare numerical vs analytical at several points
print("x   | Numerical  | Analytical (2x) | Match?")
print("-" * 50)
for x_val in [-3, -1, 0, 1, 3]:
    num = numerical_derivative(f, x_val)
    ana = 2 * x_val
    print(f"{x_val:3d} | {num:10.6f} | {ana:15.6f} | {'✓' if abs(num - ana) < 1e-4 else '✗'}")

# Works for ANY function — even ones where calculus is hard
def mystery(x):
    return np.sin(x) * x ** 2 + np.exp(-x)

print(f"\nmystery function derivative at x=1: {numerical_derivative(mystery, 1.0):.6f}")
print("(No need to do the calculus by hand!)")

### What just happened:

We computed the derivative two ways:
- **Numerical:** just measured the slope by nudging x
- **Analytical:** used the calculus rule (derivative of x² = 2x)

They match perfectly! The numerical method works for ANY function — even ugly ones where you'd never want to do calculus by hand (like the "mystery" function).

**In practice:** PyTorch uses analytical derivatives (faster, exact). But numerical derivatives are great for **checking your work** — if analytical and numerical don't match, you have a bug. This is called **gradient checking**.

## 3. Gradient Descent — The Learning Algorithm

This is **THE algorithm** that trains every neural network, every LLM, every AI model you've ever used.

### The analogy: Lost in fog on a hilly landscape

Imagine you're blindfolded on a hilly landscape and want to reach the lowest valley:
- You can't SEE the valley, but you can FEEL the slope under your feet
- Strategy: feel which way is downhill → take a step that way → repeat
- Eventually, you'll reach the bottom

That's gradient descent. The "landscape" is the loss function, and the "position" is your current weight values.

### The update rule (the most important equation in ML):

```
new_weight = old_weight - learning_rate × derivative
```

Why **subtract**? Because:
- If derivative is **positive** (slope going up) → subtracting makes weight smaller → moves left → downhill ✓
- If derivative is **negative** (slope going down) → subtracting a negative = adding → moves right → downhill ✓

Let's watch it work:

In [ ]:
# Gradient descent on f(x) = x²
# The minimum is at x=0, but let's pretend we don't know that

def f(x):
    return x ** 2

def f_deriv(x):
    return 2 * x

# Start at a random point
x = 4.0
learning_rate = 0.1
history = [(x, f(x))]

print(f"Step  0: x = {x:.4f}, f(x) = {f(x):.4f}")

for step in range(1, 21):
    gradient = f_deriv(x)
    x = x - learning_rate * gradient  # THE update rule
    history.append((x, f(x)))
    if step <= 10 or step % 5 == 0:
        print(f"Step {step:2d}: x = {x:.4f}, f(x) = {f(x):.4f}, gradient = {gradient:.4f}")

print(f"\nFinal: x ≈ {x:.6f} (should be ~0.0)")
print(f"       f(x) ≈ {f(x):.10f} (should be ~0.0)")

### Reading the output above:

Notice how:
- **x starts at 4.0** (random position on the hill)
- Each step, x gets **closer to 0** (the minimum of x²)
- The **gradient gets smaller** as we approach the bottom (slope is flatter near the minimum)
- So the steps get smaller too — it naturally slows down as it converges

This is exactly what happens during LLM training — the loss drops fast at first, then gradually levels off.

Let's visualize the path on the curve:

In [ ]:
# Visualize the gradient descent path
x_range = np.linspace(-5, 5, 100)
hist_x = [h[0] for h in history]
hist_y = [h[1] for h in history]

plt.figure(figsize=(10, 5))
plt.plot(x_range, f(x_range), 'b-', linewidth=2, label='f(x) = x²')
plt.plot(hist_x, hist_y, 'ro-', markersize=6, label='Gradient descent steps')
plt.plot(hist_x[0], hist_y[0], 'gs', markersize=12, label=f'Start (x={hist_x[0]})')
plt.plot(hist_x[-1], hist_y[-1], 'r*', markersize=15, label=f'End (x={hist_x[-1]:.3f})')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Gradient Descent: Rolling Downhill to the Minimum')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Learning Rate — The Most Important Knob in Training

The **learning rate** controls how big each step is:

```
weight = weight - learning_rate × gradient
                  ^^^^^^^^^^^^^^^
                  this is the step size multiplier
```

Choosing the right learning rate is critical:

| Learning Rate | What Happens | Analogy |
|---------------|-------------|---------|
| **Too small** (0.001) | Takes tiny baby steps, converges very slowly | An ant walking down a mountain |
| **Just right** (0.01-0.1) | Smooth, steady progress toward the minimum | A hiker taking measured steps |
| **Too large** (0.9+) | Overshoots the minimum, bounces wildly, may diverge | Trying to go downhill in 7-league boots — you leap right over the valley |

Let's see this visually:

In [ ]:
# Compare different learning rates
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x_range = np.linspace(-5, 5, 100)

for ax, lr, title in zip(axes, [0.01, 0.1, 0.9],
                           ["Too Small (0.01)\nSlow convergence",
                            "Just Right (0.1)\nSmooth convergence", 
                            "Too Large (0.9)\nOscillates wildly"]):
    x = 4.0
    path_x, path_y = [x], [f(x)]
    
    for _ in range(20):
        x = x - lr * f_deriv(x)
        path_x.append(x)
        path_y.append(f(x))
    
    ax.plot(x_range, f(x_range), 'b-', linewidth=2)
    ax.plot(path_x, path_y, 'ro-', markersize=4)
    ax.set_title(title)
    ax.set_ylim(-1, 20)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Reading the plots:

- **Left (lr=0.01):** The red dots barely move after 20 steps. It WILL eventually reach the bottom, but painfully slowly. Wasting compute time.
- **Middle (lr=0.1):** Nice smooth descent to the bottom. This is what we want.
- **Right (lr=0.9):** The dots bounce back and forth wildly! It overshoots the minimum, bounces to the other side, overshoots again. With a slightly larger learning rate, it would diverge (fly off to infinity).

### How to tell if your learning rate is right in practice:

Look at the **loss curve** (loss plotted over training steps):
- **Smooth decrease** → learning rate is good
- **Barely moving** → learning rate too small, increase it
- **Jagged / going up** → learning rate too large, decrease it
- **Fast drop then flat** → converged, training is done

Most ML practitioners start with `lr = 0.001` or `lr = 0.01` and adjust from there.

## 5. Partial Derivatives & Gradients (Multiple Variables)

So far we've optimized ONE variable (x). But real models have **millions of weights**. How do we handle that?

### Partial derivative = derivative with respect to ONE weight, ignoring the rest

If your loss depends on weights `w1` and `w2`:
```
loss(w1, w2) = w1² + w2²

∂loss/∂w1 = 2·w1    (pretend w2 doesn't exist, take derivative normally)
∂loss/∂w2 = 2·w2    (pretend w1 doesn't exist, take derivative normally)
```

### The gradient = ALL partial derivatives bundled into one vector

```
gradient = [∂loss/∂w1, ∂loss/∂w2] = [2·w1, 2·w2]
```

The gradient points in the direction of **steepest increase**. To minimize loss, we go **opposite** — steepest decrease.

### Gradient descent with multiple weights:
```python
w1 = w1 - learning_rate × ∂loss/∂w1
w2 = w2 - learning_rate × ∂loss/∂w2
```

Each weight gets its own gradient, and they ALL get updated every step. GPT-3 has 175 billion weights — each one gets updated every training step!

Let's visualize 2D gradient descent on a bowl-shaped surface:

In [ ]:
# f(x, y) = x² + y²  →  minimum at (0, 0)
# ∂f/∂x = 2x,  ∂f/∂y = 2y
# gradient = [2x, 2y]

def f_2d(x, y):
    return x**2 + y**2

def gradient_2d(x, y):
    return np.array([2*x, 2*y])

# Gradient descent in 2D
pos = np.array([4.0, 3.0])  # start position
lr = 0.1
path = [pos.copy()]

for step in range(30):
    grad = gradient_2d(pos[0], pos[1])
    pos = pos - lr * grad
    path.append(pos.copy())

path = np.array(path)

# Visualize with contour plot
fig, ax = plt.subplots(figsize=(8, 8))
x_grid = np.linspace(-5, 5, 100)
y_grid = np.linspace(-5, 5, 100)
X, Y = np.meshgrid(x_grid, y_grid)
Z = f_2d(X, Y)

ax.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.6)
ax.plot(path[:, 0], path[:, 1], 'ro-', markersize=4, label='Gradient descent path')
ax.plot(path[0, 0], path[0, 1], 'gs', markersize=12, label='Start')
ax.plot(path[-1, 0], path[-1, 1], 'r*', markersize=15, label='End')
ax.set_xlabel('x (weight 1)')
ax.set_ylabel('y (weight 2)')
ax.set_title('2D Gradient Descent: Rolling Down a Bowl')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.show()

print(f"Started at: ({path[0, 0]:.1f}, {path[0, 1]:.1f})")
print(f"Ended at:   ({path[-1, 0]:.4f}, {path[-1, 1]:.4f})")
print(f"Minimum is at (0, 0)")

### Reading the contour plot:

- The **rings** are like a topographic map — each ring is a constant height (loss value). The center (0,0) is the lowest point.
- The **red path** shows gradient descent walking downhill from (4, 3) to (0, 0)
- Notice it takes a **diagonal path** — it's simultaneously reducing both w1 and w2
- Each step follows the steepest downhill direction at that point

**This is exactly what happens during training** — but instead of 2 weights on a 2D surface, you have millions of weights in a million-dimensional landscape. The math is identical, just scaled up.

## 6. Chain Rule — Why Backpropagation Works

Here's the problem: a neural network isn't just one function — it's a **chain of functions stacked together**:

```
input → [layer 1] → [layer 2] → [layer 3] → prediction → loss
         weights1     weights2     weights3
```

To update `weights1`, we need to know: **"how does changing weights1 affect the final loss?"**

But weights1 doesn't directly touch the loss! It affects layer 1's output, which affects layer 2's output, which affects layer 3's output, which affects the loss. There's a chain of dependencies.

### The chain rule solves this by multiplying through:

```
Simple example: y = (3x + 2)²

Break it into steps:
  Step 1: a = 3x + 2       (inner function)
  Step 2: y = a²            (outer function)

How does y change when x changes?
  dy/dx = dy/da × da/dx
        = 2a    × 3
        = 2(3x+2) × 3
        = 6(3x+2)
```

**For neural networks:**
```
∂loss/∂weights1 = (∂loss/∂layer3) × (∂layer3/∂layer2) × (∂layer2/∂layer1) × (∂layer1/∂weights1)
```

This backward multiplication is called **backpropagation** — we propagate the gradient backwards through each layer.

Let's verify the chain rule works with code:

In [ ]:
# Chain rule example: y = (3x + 2)²
# Break it down:  h(x) = 3x + 2,  f(h) = h²
# dy/dx = f'(h) × h'(x) = 2h × 3 = 2(3x+2) × 3 = 6(3x+2)

def y(x):
    return (3 * x + 2) ** 2

def dy_dx_analytical(x):
    return 6 * (3 * x + 2)

# Verify with numerical derivative
x_test = 1.0
num = numerical_derivative(y, x_test)
ana = dy_dx_analytical(x_test)
print(f"y = (3x + 2)² at x = {x_test}")
print(f"Numerical derivative:  {num:.6f}")
print(f"Analytical (chain rule): {ana:.6f}")
print(f"Match: {'✓' if abs(num - ana) < 1e-4 else '✗'}")

# Now a 3-layer chain: y = sin(exp(x²))
# h1(x) = x²     →  h1' = 2x
# h2(z) = exp(z)  →  h2' = exp(z)
# h3(z) = sin(z)  →  h3' = cos(z)
# dy/dx = cos(exp(x²)) × exp(x²) × 2x

def y_chain(x):
    return np.sin(np.exp(x ** 2))

def dy_chain_analytical(x):
    return np.cos(np.exp(x**2)) * np.exp(x**2) * 2 * x

x_test = 0.5
num = numerical_derivative(y_chain, x_test)
ana = dy_chain_analytical(x_test)
print(f"\ny = sin(exp(x²)) at x = {x_test}")
print(f"Numerical:  {num:.6f}")
print(f"Chain rule: {ana:.6f}")
print(f"Match: {'✓' if abs(num - ana) < 1e-4 else '✗'}")
print("\nThis is EXACTLY what backpropagation does — chain rule through every layer!")

### What just happened:

We verified the chain rule on two functions:
1. **Simple:** `y = (3x + 2)²` — a 2-step chain
2. **Complex:** `y = sin(exp(x²))` — a 3-step chain (like a 3-layer network!)

In both cases, the chain rule (multiply derivatives of each step) gave the exact same answer as the numerical "just measure it" approach. ✓

**The key insight:** No matter how many layers deep your network is, you can always compute how any weight affects the final loss — just multiply the derivatives backwards through each layer. This is backpropagation, and it's what makes training deep neural networks possible.

**Tomorrow in Day 03:** You'll see that PyTorch does all of this automatically. You just write the forward pass, call `loss.backward()`, and PyTorch computes every gradient via the chain rule for you. Magic!

## 7. Mini Project: Train a Line to Fit Data (Linear Regression from Scratch)

Time to put **everything together**. We'll:
1. Generate some data that follows `y = 3x + 7` (with noise)
2. Start with random weight `w` and bias `b`
3. Use gradient descent to learn the correct `w ≈ 3` and `b ≈ 7`

This is exactly how a single neuron learns — and a neural network is just many neurons stacked together.

### What each step does:

```python
# FORWARD PASS: make a prediction
y_pred = w * X + b

# LOSS: how wrong are we? (Mean Squared Error)
loss = mean((y_pred - y_true)²)

# GRADIENTS: which direction should w and b move?
dw = mean(2 * (y_pred - y_true) * X)   # partial derivative w.r.t. w
db = mean(2 * (y_pred - y_true))        # partial derivative w.r.t. b

# UPDATE: nudge w and b in the right direction
w = w - learning_rate * dw
b = b - learning_rate * db
```

### Where do those gradient formulas come from?

```
loss = mean((w*X + b - y_true)²)

Using the chain rule:
  ∂loss/∂w = mean(2 * (w*X + b - y_true) * X)     ← the X at the end comes from ∂(w*X)/∂w = X
  ∂loss/∂b = mean(2 * (w*X + b - y_true) * 1)      ← the 1 comes from ∂(b)/∂b = 1
```

Let's watch it learn:

In [ ]:
# Generate some noisy data: y = 3x + 7 (true relationship)
np.random.seed(42)
X = np.random.uniform(-5, 5, 50)
y_true = 3 * X + 7 + np.random.randn(50) * 2  # add noise

# Initialize random weight and bias
w = np.random.randn()  # should learn → 3
b = np.random.randn()  # should learn → 7
learning_rate = 0.01

losses = []

print(f"Starting: w={w:.3f} (target: 3.0), b={b:.3f} (target: 7.0)\n")

for epoch in range(100):
    # Forward pass: predictions
    y_pred = w * X + b
    
    # Loss: Mean Squared Error = average of (prediction - actual)²
    loss = np.mean((y_pred - y_true) ** 2)
    losses.append(loss)
    
    # Gradients (partial derivatives of loss w.r.t. w and b)
    # d(loss)/dw = mean(2 * (y_pred - y_true) * X)
    # d(loss)/db = mean(2 * (y_pred - y_true))
    dw = np.mean(2 * (y_pred - y_true) * X)
    db = np.mean(2 * (y_pred - y_true))
    
    # Update weights
    w = w - learning_rate * dw
    b = b - learning_rate * db
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}: loss={loss:.3f}, w={w:.3f}, b={b:.3f}")

print(f"\nFinal:  w={w:.3f} (target: 3.0), b={b:.3f} (target: 7.0)")
print(f"The model LEARNED the relationship from data!")

### Reading the output:

Look at how `w` and `b` evolve:
- They start as random numbers (far from the targets of 3.0 and 7.0)
- Each epoch, they get a little closer to the true values
- The **loss drops** every epoch — the model is getting less wrong
- By epoch 90-100, `w ≈ 3.0` and `b ≈ 7.0` — it learned the relationship!

**This is training.** The model started knowing nothing, and by repeatedly measuring "how wrong am I?" and adjusting, it figured out the pattern.

Now let's see this visually:

In [ ]:
# Visualize: the learned line + loss curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: data + learned line
ax1.scatter(X, y_true, alpha=0.6, label='Data points')
x_line = np.linspace(-5, 5, 100)
ax1.plot(x_line, w * x_line + b, 'r-', linewidth=2, 
         label=f'Learned: y = {w:.2f}x + {b:.2f}')
ax1.plot(x_line, 3 * x_line + 7, 'g--', linewidth=2, 
         label='True: y = 3x + 7', alpha=0.5)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title('Learned Line vs True Line')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: loss over time
ax2.plot(losses, 'b-', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss (MSE)')
ax2.set_title('Loss Curve — Model Getting Better Over Time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The loss curve going down = the model is learning!")
print("This same pattern applies to training GPT — just with billions of weights.")

### What the plots show:

**Left plot (Learned Line vs True Line):**
- Blue dots = the noisy training data
- Green dashed line = the true relationship (y = 3x + 7) that we're trying to find
- Red line = what the model learned — it's almost on top of the green line!
- The model found the pattern **from noisy data alone**, without ever being told the formula

**Right plot (Loss Curve):**
- This is the most important plot in all of ML
- Loss starts high (model is random, predictions are terrible)
- Loss drops quickly at first, then levels off (most learning happens early)
- When the curve flattens, the model has converged — further training won't help much

**Connecting to LLMs:**
- Replace `y = wx + b` with a 175-billion-parameter transformer
- Replace our 50 data points with trillions of tokens from the internet
- The training loop is identical: forward pass → loss → gradients → update
- The loss curve looks the same: starts high, drops, levels off
- Training GPT-4 probably took months of this loop running on thousands of GPUs

## Exercises

1. **Learning rate experiment:** Re-run the linear regression with `learning_rate = 0.001` and `learning_rate = 0.5`. Plot the loss curves — what happens?

2. **More variables:** Modify the mini project to learn `y = 2x₁ + 5x₂ - 3`. You'll need 2 weights and 1 bias. (Hint: generate 2D input data)

3. **Numerical gradient check:** For the linear regression, compute dw and db numerically (nudge w by h, see how loss changes). Verify they match the analytical gradients.

4. **Non-linear function:** Try gradient descent on `f(x) = x⁴ - 3x³ + 2`. Does it find the global minimum? (Hint: it might get stuck in a local minimum — this is a real problem in ML!)

---

## Key Takeaways

- **Derivative** = slope = "which way is downhill?"
- **Gradient** = vector of all partial derivatives (one per weight)
- **Gradient descent** = take small steps opposite to the gradient → minimize loss
- **Learning rate** = step size. Too big → chaos. Too small → slow. Just right → convergence
- **Chain rule** = multiply derivatives through each layer → backpropagation
- **Loss curve going down** = the model is learning
- Tomorrow: PyTorch does ALL of this automatically with **autograd**!